# non-diff-fn-wrap — worked example 3: Global tracking toggle does not override is_differentiable=False

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `non-diff-fn-wrap`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The `grad_tracking_enabled` global toggle can suppress Recipe creation even for differentiable ops, but it cannot *enable* gradient flow through non-differentiable ops. When `is_differentiable=False`, the output will always have `requires_grad=False` and `recipe=None`, regardless of whether global tracking is on or off. The three gates are ANDed: any False makes the whole expression False.

## Worked solution

**Step 1 — Wrap a differentiable and a non-differentiable op.**
We use `torch.add` (differentiable) and `torch.eq` (non-differentiable).

**Step 2 — Case A: tracking on, non-diff op.**
With `grad_tracking_enabled=True` and `is_differentiable=False`, the AND is False. `eq_wrap(x, y).requires_grad` is False.

**Step 3 — Case B: tracking off, differentiable op.**
With `grad_tracking_enabled=False` and `is_differentiable=True`, the AND is also False. `add_wrap(x, y).requires_grad` is False because tracking is off.

**Step 4 — Case C: tracking on, differentiable op.**
Now the AND is True (tracking on AND differentiable AND inputs tracked). `add_wrap(x, y).requires_grad` is True.

**Step 5 — Interpret the results.**
The three-gate AND means both gates must be True. Non-differentiable ops are permanently excluded regardless of the toggle.

In [ ]:
import torch

# ---- Minimal scaffold ----
class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args
        self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = None

grad_tracking_enabled = True   # module-level toggle

def wrap_forward_fn(fwd_fn, is_differentiable: bool = True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        # Read toggle fresh each call (supports mid-run toggling)
        on = globals()['grad_tracking_enabled']
        requires_grad = (
            on
            and is_differentiable
            and any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        )
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

add_wrap = wrap_forward_fn(torch.add)
eq_wrap  = wrap_forward_fn(torch.eq,  is_differentiable=False)

x = MiniTensor(torch.tensor([1.0, 2.0]), requires_grad=True)
y = MiniTensor(torch.tensor([1.0, 3.0]), requires_grad=True)

# Case A: tracking ON, non-diff op
grad_tracking_enabled = True
out_eq = eq_wrap(x, y)
print(f'Case A (eq, tracking=on ) requires_grad: {out_eq.requires_grad}')  # False

# Case B: tracking OFF, diff op
grad_tracking_enabled = False
out_add_off = add_wrap(x, y)
print(f'Case B (add, tracking=off) requires_grad: {out_add_off.requires_grad}')  # False

# Case C: tracking ON, diff op
grad_tracking_enabled = True
out_add_on = add_wrap(x, y)
print(f'Case C (add, tracking=on ) requires_grad: {out_add_on.requires_grad}')   # True

assert out_eq.requires_grad      is False
assert out_add_off.requires_grad is False
assert out_add_on.requires_grad  is True
grad_tracking_enabled = True  # restore